# Phase 2 — Information Richness Ablation (Leakage-Free)

Ömer Faruk Merey — Middle East Technical University

**Research question.** *How does the semantic richness of auxiliary text captions — ranging from a vision-only baseline, to simple keywords, to detailed qualitative descriptions — impact the accuracy of land cover composition regression from satellite images while strictly avoiding data leakage?*

**Five conditions** (5 × 3 seeds = 15 runs):

| Code | Architecture | Text input |
|---|---|---|
| R0a | CLIP + cross-attn + MLP | `""` (empty token) |
| R0b | CLIP + GAP + MLP | — (no text path) |
| R1  | CLIP + cross-attn + MLP | NER(vision_gemma3 ∪ vision_qwen) keyword string |
| R2a | CLIP + cross-attn + MLP | masked `vision_gemma3-4b` |
| R2b | CLIP + cross-attn + MLP | masked `vision_qwen3-vl-8b` |

Encoders are **frozen**; only fusion + head train.

**M-series Mac note.** CLIP is frozen, so its forward is identical every epoch. We precompute patch + text embeddings once and feed those tensors into training. Same final weights, ~5–10× wall-time speedup vs. running CLIP per batch.


## 1. Setup

In [1]:
%pip uninstall clip -y -q
%pip install -q git+https://github.com/openai/CLIP.git spacy wandb tqdm scipy
%pip install -q --upgrade torch torchvision
!python -m spacy download en_core_web_sm -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import sys, os
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('repo root:', REPO)

import torch
import clip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.sanitize import mask_numbers, extract_keywords
from src.dataset import (
    ARASDataset, PrecomputedARASDataset, CONDITIONS, COMPOSITION_CLASSES,
)
from src.precompute import precompute_all
from src.model import build_model
from src.train import TrainConfig, train_one_condition
from src.eval import predict_split, compute_metrics, aggregate_runs, plot_attention

DATASET_DIR = REPO / 'dataset'
IMAGES_DIR = DATASET_DIR / 'images'
CAPTIONS_CSV = DATASET_DIR / 'captions.csv'
CKPT_DIR = REPO / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
PRECOMPUTED_DIR = REPO / 'precomputed'; PRECOMPUTED_DIR.mkdir(exist_ok=True)
REPORTS_DIR = REPO / 'reports' / 'phase-2'; REPORTS_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', device)

clip_model, preprocess = clip.load('ViT-B/32', device=device)
clip_model.eval()
tokenizer = clip.tokenize

repo root: /Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition


/opt/anaconda3/envs/mp311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 2. Sanity checks on sanitization & keyword extraction

In [3]:
demo = 'The image shows grasslands (92%) and crops 2.5% with 6 trees.'
print('original :', demo)
print('masked   :', mask_numbers(demo))

df = pd.read_csv(CAPTIONS_CSV)
row = df.iloc[0]
print('\n--- vision_gemma3-4b ---'); print(row['vision_gemma3-4b'])
print('\n--- vision_qwen3-vl-8b ---'); print(row['vision_qwen3-vl-8b'])
kw = extract_keywords([row['vision_gemma3-4b'], row['vision_qwen3-vl-8b']])
print('\n--- R1 keywords ---'); print(kw)

original : The image shows grasslands (92%) and crops 2.5% with 6 trees.
masked   : The image shows grasslands ([NUM]) and crops [NUM] with [NUM] trees.

--- vision_gemma3-4b ---
This image depicts a rugged, arid landscape characterized by extensive rocky terrain and deeply incised gullies, likely indicative of a desert or semi-arid environment with minimal vegetation cover. The dominant land use appears to be bare rock and soil, shaped by erosion and geological processes.

--- vision_qwen3-vl-8b ---
This remote sensing image shows a rugged, arid landscape dominated by exposed bedrock and sparse vegetation, with a prominent linear feature—likely a dry riverbed or fault line—cutting through the terrain, indicating a geologically active or erosion-prone region.

--- R1 keywords ---
bedrock, cover, desert, environment, feature, image, landscape, line, process, region, riverbed, rock, soil, terrain, use, vegetation


## 3. Inspect a sample for each condition (online dataset)

In [4]:
for c in CONDITIONS:
    ds = ARASDataset(CAPTIONS_CSV, IMAGES_DIR, 'train', c, preprocess, tokenizer,
                     precompute_text=False)
    sample = ds[0]
    txt = sample.get('text', '<no text path>')
    print(f'[{c}] image={tuple(sample["image"].shape)}  gt_sum={sample["gt"].sum():.2f}')
    print(f'      text: {txt[:140]!r}')
    print()

[R0a] image=(3, 224, 224)  gt_sum=1.00
      text: ''

[R0b] image=(3, 224, 224)  gt_sum=1.00
      text: '<no text path>'

[R1] image=(3, 224, 224)  gt_sum=1.00
      text: 'area, dune, image, landscape, moisture, outcrop, patch, pattern, presence, region, river, scene, stream, terrain, that, valley, vegetation, '

[R2a] image=(3, 224, 224)  gt_sum=1.00
      text: 'This image depicts a heavily eroded, arid landscape likely within a desert or semi-arid region, characterized by extensive sand dunes and ro'

[R2b] image=(3, 224, 224)  gt_sum=1.00
      text: 'This remote sensing image shows a semi-arid or arid landscape dominated by dry, brown terrain with sparse vegetation, featuring a distinct, '



## 4. Precompute frozen CLIP embeddings (one-time, ~3–5 min on M4)

Writes:
- `precomputed/images.pt`            — (10000, 49, 512) fp16
- `precomputed/text_R0a.pt`          — (10000, 77, 512) fp16  (empty captions)
- `precomputed/text_R1.pt`           — (10000, 77, 512) fp16  (NER keywords)
- `precomputed/text_R2a.pt`          — (10000, 77, 512) fp16  (masked vision_gemma)
- `precomputed/text_R2b.pt`          — (10000, 77, 512) fp16  (masked vision_qwen)
- `precomputed/filenames.json`       — ordering check

Total: ~3.7 GB on disk. Re-runs skip already-existing files.

In [5]:
precompute_all(
    CAPTIONS_CSV, IMAGES_DIR, PRECOMPUTED_DIR,
    clip_model, preprocess, tokenizer, device,
    overwrite=False,
)
list(PRECOMPUTED_DIR.iterdir())

image embed: 100%|██████████| 157/157 [00:41<00:00,  3.74it/s]


[done] images: (10000, 49, 512) -> /Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/images.pt


text R0a: 100%|██████████| 40/40 [00:15<00:00,  2.64it/s]


[done] text R0a: (10000, 77, 512) -> /Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/text_R0a.pt


text R1: 100%|██████████| 40/40 [00:15<00:00,  2.60it/s]


...

text R2a: 100%|██████████| 40/40 [00:15<00:00,  2.55it/s]


[done] text R2a: (10000, 77, 512) -> /Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/text_R2a.pt


text R2b: 100%|██████████| 40/40 [00:15<00:00,  2.56it/s]


[done] text R2b: (10000, 77, 512) -> /Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/text_R2b.pt


[PosixPath('/Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/text_R2b.pt'),
 PosixPath('/Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/images.pt'),
 PosixPath('/Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/text_R1.pt'),
 PosixPath('/Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/filenames.json'),
 PosixPath('/Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/text_R2a.pt'),
 PosixPath('/Users/ofm/Desktop/my_repostation/remote-sensing-caption-composition/precomputed/text_R0a.pt')]

## 5. Smoke test (1 epoch, R0a, no W&B) — uses precomputed embeddings

In [6]:
smoke_cfg = TrainConfig(
    condition='R0a', seed=42, epochs=1, batch_size=64,
    use_wandb=False, num_workers=0,
    checkpoint_dir=str(CKPT_DIR / 'smoke'),
    precomputed_dir=str(PRECOMPUTED_DIR),
)
smoke = train_one_condition(
    smoke_cfg, CAPTIONS_CSV, IMAGES_DIR, clip_model, preprocess, tokenizer, device,
)
smoke['history']

[R0a s42] ep00 train=0.04147 val=0.01649 best=0.01649@0 (5.0s)


[{'epoch': 0,
  'train_loss': 0.041470489195414954,
  'val_loss': 0.016486122707525887,
  'lr': 0.0,
  'elapsed_s': 4.958010911941528}]

## 6. Full ablation grid (5 conditions × 3 seeds, precomputed)

Resume-friendly: skips conditions/seeds that already have a checkpoint.

In [ ]:
USE_WANDB = True
EPOCHS = 30
SEEDS = [42, 1337, 2024]
BATCH_SIZE = 128  # precomputed mode is light enough for big batches

all_results = {}
for cond in CONDITIONS:
    for seed in SEEDS:
        cfg = TrainConfig(
            condition=cond, seed=seed, epochs=EPOCHS, batch_size=BATCH_SIZE,
            use_wandb=USE_WANDB,
            checkpoint_dir=str(CKPT_DIR),
            precomputed_dir=str(PRECOMPUTED_DIR),
        )
        ckpt = CKPT_DIR / f'{cond}_seed{seed}_best.pt'
        if ckpt.exists():
            print(f'-- skipping {cond} seed={seed}: ckpt exists at {ckpt}')
            continue
        print(f'\n=== train {cond} seed={seed} ===')
        all_results[(cond, seed)] = train_one_condition(
            cfg, CAPTIONS_CSV, IMAGES_DIR, clip_model, preprocess, tokenizer, device,
        )


=== train R0a seed=42 ===


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/ofm/.netrc.
wandb: Currently logged in as: mereyomerfaruk (team-lingua) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[R0a s42] ep00 train=0.05322 val=0.04939 best=0.04939@0 (5.7s)
[R0a s42] ep01 train=0.02556 val=0.01605 best=0.01605@1 (3.9s)
[R0a s42] ep02 train=0.01410 val=0.01403 best=0.01403@2 (4.0s)
[R0a s42] ep03 train=0.01290 val=0.01293 best=0.01293@3 (3.8s)
[R0a s42] ep04 train=0.01199 val=0.01245 best=0.01245@4 (3.9s)
[R0a s42] ep05 train=0.01124 val=0.01146 best=0.01146@5 (3.8s)
[R0a s42] ep06 train=0.00982 val=0.00978 best=0.00978@6 (3.8s)
[R0a s42] ep07 train=0.00879 val=0.00902 best=0.00902@7 (3.9s)
[R0a s42] ep08 train=0.00837 val=0.00885 best=0.00885@8 (3.9s)
[R0a s42] ep09 train=0.00785 val=0.00851 best=0.00851@9 (3.9s)
[R0a s42] ep10 train=0.00765 val=0.00849 best=0.00849@10 (3.8s)
[R0a s42] ep11 train=0.00747 val=0.00821 best=0.00821@11 (3.9s)
[R0a s42] ep12 train=0.00725 val=0.00802 best=0.00802@12 (3.8s)
[R0a s42] ep13 train=0.00703 val=0.00827 best=0.00802@12 (4.1s)
[R0a s42] ep14 train=0.00687 val=0.00800 best=0.00800@14 (4.0s)
[R0a s42] ep15 train=0.00682 val=0.00769 best=0.00

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


[R0a s42] ep29 train=0.00605 val=0.00736 best=0.00735@27 (3.8s)


...

[R2a s42] ep00 train=0.03748 val=0.02366 best=0.02366@0 (5.9s)
[R2a s42] ep01 train=0.02159 val=0.01939 best=0.01939@1 (4.8s)
[R2a s42] ep02 train=0.01788 val=0.01590 best=0.01590@2 (4.9s)
[R2a s42] ep03 train=0.01459 val=0.01364 best=0.01364@3 (4.9s)
[R2a s42] ep04 train=0.01244 val=0.01201 best=0.01201@4 (5.2s)
[R2a s42] ep05 train=0.01081 val=0.01091 best=0.01091@5 (5.1s)
[R2a s42] ep06 train=0.00991 val=0.01025 best=0.01025@6 (4.8s)
[R2a s42] ep07 train=0.00944 val=0.01008 best=0.01008@7 (4.9s)
[R2a s42] ep08 train=0.00878 val=0.00934 best=0.00934@8 (5.3s)
[R2a s42] ep09 train=0.00811 val=0.00918 best=0.00918@9 (4.8s)
[R2a s42] ep10 train=0.00778 val=0.00904 best=0.00904@10 (5.0s)
[R2a s42] ep11 train=0.00751 val=0.00859 best=0.00859@11 (5.2s)
[R2a s42] ep12 train=0.00718 val=0.00844 best=0.00844@12 (4.9s)
[R2a s42] ep13 train=0.00700 val=0.00838 best=0.00838@13 (4.9s)
[R2a s42] ep14 train=0.00678 val=0.00818 best=0.00818@14 (4.8s)
[R2a s42] ep15 train=0.00666 val=0.00807 best=0.00

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


[R2a s42] ep29 train=0.00580 val=0.00769 best=0.00769@27 (4.7s)


elapsed_s,█▃▄▃▅▅▃▄▅▃▄▅▄▄▃▃▃▃▃▃▂▄▁▃▂▂▃▄▃▃
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▄▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▆▅▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_epoch,27
best_val_loss,0.00769
elapsed_s,4.69507
epoch,29
lr,0
train_loss,0.0058



=== train R2a seed=1337 ===


## 7. Test set evaluation

In [ ]:
rows = []
predictions = {}
for cond in CONDITIONS:
    cond_metrics = []
    for seed in SEEDS:
        ckpt = CKPT_DIR / f'{cond}_seed{seed}_best.pt'
        if not ckpt.exists():
            print(f'!! missing ckpt: {ckpt}')
            continue
        out = predict_split(
            cond, seed, CAPTIONS_CSV, IMAGES_DIR,
            clip_model, preprocess, tokenizer, device, ckpt, split='test',
            precomputed_dir=str(PRECOMPUTED_DIR), batch_size=BATCH_SIZE,
        )
        m = compute_metrics(out['pred'], out['gt'])
        m['condition'] = cond; m['seed'] = seed
        cond_metrics.append(m)
        predictions[(cond, seed)] = out
        rows.append(m)
    if cond_metrics:
        agg = aggregate_runs(cond_metrics)
        print(f'{cond}: MSE {agg["mse_mean"]:.5f} ± {agg["mse_std"]:.5f}  '
              f'MAE {agg["mae_mean"]:.5f} ± {agg["mae_std"]:.5f}')

results_df = pd.DataFrame(rows)
results_df.to_csv(REPORTS_DIR / 'phase2_results.csv', index=False)
results_df

## 8. Aggregated table & paired significance test

In [ ]:
summary = (results_df
           .groupby('condition')[['mse','mae','rmse','kl','sum_violation']]
           .agg(['mean','std']))
summary

In [ ]:
from scipy.stats import ttest_rel

def per_sample_mse(cond):
    arrs = []
    for seed in SEEDS:
        if (cond, seed) not in predictions:
            continue
        o = predictions[(cond, seed)]
        arrs.append(((o['pred'] - o['gt'])**2).mean(axis=-1))
    return np.mean(np.stack(arrs), axis=0) if arrs else None

pairs = [('R0a','R1'), ('R0a','R2a'), ('R0a','R2b'),
         ('R0b','R1'), ('R0b','R2a'), ('R0b','R2b'),
         ('R1','R2a'), ('R1','R2b')]
for a, b in pairs:
    pa, pb = per_sample_mse(a), per_sample_mse(b)
    if pa is None or pb is None:
        continue
    t, p = ttest_rel(pa, pb)
    print(f'{a:>4s} vs {b:<4s}  t={t:+.3f}  p={p:.4g}  '
          f'(mean diff = {pa.mean()-pb.mean():+.5f})')

## 9. Bar chart with error bars

In [ ]:
means = summary['mse']['mean']; stds = summary['mse']['std']
order = [c for c in CONDITIONS if c in means.index]

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(order, [means[c] for c in order], yerr=[stds[c] for c in order], capsize=4)
ax.set_ylabel('Test MSE'); ax.set_title('Information Richness Ablation')
plt.tight_layout(); plt.savefig(REPORTS_DIR / 'phase2_mse_bar.png', dpi=150)
plt.show()

## 10. Cross-attention visualisation (R2a, online forward for raw images)

For viz only we re-run CLIP on the few sample images (precomputed mode discards intermediate attention).

In [ ]:
ckpt = CKPT_DIR / 'R2a_seed42_best.pt'
if ckpt.exists():
    model = build_model('R2a', clip_model).to(device)
    state = torch.load(ckpt, map_location=device)
    model.load_state_dict(state['state_dict'], strict=False)
    model.eval()

    test_ds = ARASDataset(CAPTIONS_CSV, IMAGES_DIR, 'test', 'R2a',
                          preprocess, tokenizer, precompute_text=False)
    fig, axes = plt.subplots(1, 4, figsize=(16,4))
    for k, idx in enumerate(np.linspace(0, len(test_ds)-1, 4, dtype=int)):
        sample = test_ds[idx]
        with torch.no_grad():
            pred, attn = model(sample['image'].unsqueeze(0).to(device),
                                sample['tokens'].unsqueeze(0).to(device),
                                return_attention=True)
        img = Image.open(IMAGES_DIR / sample['filename']).convert('RGB').resize((224,224))
        plot_attention(img, attn.cpu().numpy(), ax=axes[k])
        axes[k].set_title(sample['filename'], fontsize=9)
    plt.tight_layout(); plt.savefig(REPORTS_DIR / 'phase2_attention.png', dpi=150)
    plt.show()
else:
    print('R2a checkpoint not found yet — run the training grid first.')